# XplainTDA

In [1]:
import scanpy as sc
from XplainTDA import XplainTDA

❌ CUDA not available. Using CPU.


In [2]:
adata=sc.read_h5ad("Datasets/PBMC/CM/PBMC_GEX_Def.h5ad")
adata

AnnData object with n_obs × n_vars = 5639 × 2752
    obs: 'RNA_celltype', 'Raw_CellType', 'CellType', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'n_genes_by_counts', 'total_counts', 'total_counts_MT', 'pct_counts_MT', 'doublet_score', 'predicted_doublet', 'passes_filter'
    var: 'gene_ids', 'feature_types', 'n_cells', 'log_n_cells', 'MT', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'scrublet'
    obsm: 'X_GRAE'

In [3]:
adata_red=adata[:100, :30].copy()
adata_red

AnnData object with n_obs × n_vars = 100 × 30
    obs: 'RNA_celltype', 'Raw_CellType', 'CellType', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'n_genes_by_counts', 'total_counts', 'total_counts_MT', 'pct_counts_MT', 'doublet_score', 'predicted_doublet', 'passes_filter'
    var: 'gene_ids', 'feature_types', 'n_cells', 'log_n_cells', 'MT', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'scrublet'
    obsm: 'X_GRAE'

In [4]:
tda = XplainTDA(adata_red, embedding_key="X_GRAE")
tda.fit()
tda.save("tda.joblib")

--------------I AM USING GUDHI--------------


In [5]:
tda2 = XplainTDA.load("tda.joblib")
tda2

# Local_XplainTDA

In [1]:
from pathlib import Path

import scanpy as sc

from XplainTDA import XplainTDA
from Local_XplainTDA import Local_XplainTDA


# -----------------------------------------------------
# Paths
# -----------------------------------------------------
adata_path = Path("Datasets/PBMC/CM/PBMC_GEX_Def.h5ad")
grae_path = Path("Datasets/PBMC/CM/PBMC_GEX_GRAE_embedder.joblib")


# -----------------------------------------------------
# Load data and verify required files
# -----------------------------------------------------
assert adata_path.exists(), f"Missing file: {adata_path}"
assert grae_path.exists(), f"Missing file: {grae_path}"

adata = sc.read_h5ad(adata_path)

assert "X_GRAE" in adata.obsm, (
    "The AnnData object does not contain adata.obsm['X_GRAE']."
)

❌ CUDA not available. Using CPU.


In [2]:
# -----------------------------------------------------
# Build TDA from the already-computed embedding
# -----------------------------------------------------
tda = XplainTDA(
    adata=adata,
    embedding_key="X_GRAE",
    maxdim=0,
).fit()

--------------I AM USING GUDHI--------------


In [3]:
# Load and attach the saved GRAE model to the existing TDA object.
explainer3 = Local_XplainTDA(
    tda=tda,
    grae_model_path=grae_path,
    embedding_key="X_GRAE",
    target_dim=0
)

assert tda.embedder is not None

In [ ]:
# Test three genes on at most one localized H1 feature.
genes = adata.var_names[:3].tolist()

results3 = explainer3.importance_for_genes(
    genes=genes,
    n_jobs=3)

results3

Launching 3 jobs | H0 | 30 features Mon, 15 Jun 2026 10:10:34
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
Done H0 feature 5636 | 5570 cells Mon, 15 Jun 2026 12:18:44
Done H0 feature 5637 | 5639 cells Mon, 15 Jun 2026 12:18:45
Done H0 feature 5638 | 5639 cells Mon, 15 Jun 2026 12:18:52
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------
--------------I AM USING GUDHI--------------


# Global_XplainTDA

In [1]:
from pathlib import Path

import scanpy as sc

from XplainTDA import XplainTDA
from Global_XplainTDA import Global_XplainTDA


# -----------------------------------------------------
# Paths
# -----------------------------------------------------
adata_path = Path("Datasets/PBMC/CM/PBMC_GEX_Def.h5ad")
grae_path = Path("Datasets/PBMC/CM/PBMC_GEX_GRAE_embedder.joblib")


# -----------------------------------------------------
# Load data and verify required files
# -----------------------------------------------------
assert adata_path.exists(), f"Missing file: {adata_path}"
assert grae_path.exists(), f"Missing file: {grae_path}"

adata = sc.read_h5ad(adata_path)

assert "X_GRAE" in adata.obsm, (
    "The AnnData object does not contain adata.obsm['X_GRAE']."
)


❌ CUDA not available. Using CPU.


In [3]:
# -----------------------------------------------------
# Build TDA from the already-computed embedding
# -----------------------------------------------------
tda = XplainTDA(
    adata=adata,
    embedding_key="X_GRAE",
    n_neighbors=75,
    maxdim=0,
    ph_backend="ripser",
).fit()

--------------I AM USING RIEPSR--------------


In [4]:
# -----------------------------------------------------
# Load the trained GRAE inside the global explainer
# -----------------------------------------------------
explainer = Global_XplainTDA(
    tda=tda,
    grae_model_path=grae_path,
    target_dim=0,
    ph_backend="ripser",
    perturbation="zero",
    distance="bottleneck",
    verbose=True,
)

assert tda.embedder is not None, "The saved GRAE model was not attached to tda."

In [14]:
# -----------------------------------------------------
# Test only three genes and one worker
# -----------------------------------------------------
genes = adata.var_names[:6].tolist()

results = explainer.importance_by_homology(
    target_dim=0,
    genes=genes,
    n_jobs=3,
)

results

,homology_dim,gene_idx,gene,ph_backend,distance_metric,score,approximation,approximation_note,rank
0,0,2,C1orf159,ripser,bottleneck,0.004390,True,Ripser on sparse kNN distance matrix,1
1,0,3,TNFRSF4,ripser,bottleneck,0.002189,True,Ripser on sparse kNN distance matrix,2
2,0,0,PLEKHN1,ripser,bottleneck,0.002183,True,Ripser on sparse kNN distance matrix,3
3,0,1,HES4,ripser,bottleneck,0.002062,True,Ripser on sparse kNN distance matrix,4
4,0,5,AJAP1,ripser,bottleneck,0.001249,True,Ripser on sparse kNN distance matrix,5
5,0,4,SMIM1,ripser,bottleneck,0.000274,True,Ripser on sparse kNN distance matrix,6


In [15]:
# -----------------------------------------------------
# Load the trained GRAE inside the global explainer
# -----------------------------------------------------
explainer2 = Global_XplainTDA(
    tda=tda,
    target_dim=0,
    ph_backend="ripser",
    perturbation="zero",
    distance="bottleneck",
    verbose=True,
)

In [13]:
# -----------------------------------------------------
# Test only three genes and one worker
# -----------------------------------------------------
genes = adata.var_names[:6].tolist()

results2 = explainer2.importance_by_homology(
    target_dim=0,
    genes=genes,
    n_jobs=3,
)

results2

Launching 3 jobs | genes=6 | dims=[0] | ph_backend=ripser
Done gene PLEKHN1 (0) | dims=[0] | Mon, 15 Jun 2026 09:44:20
Done gene C1orf159 (2) | dims=[0] | Mon, 15 Jun 2026 09:44:21
Done gene HES4 (1) | dims=[0] | Mon, 15 Jun 2026 09:44:22
Done gene TNFRSF4 (3) | dims=[0] | Mon, 15 Jun 2026 09:44:32
Done gene SMIM1 (4) | dims=[0] | Mon, 15 Jun 2026 09:44:34
Done gene AJAP1 (5) | dims=[0] | Mon, 15 Jun 2026 09:44:35


,homology_dim,gene_idx,gene,ph_backend,distance_metric,score,approximation,approximation_note,rank
0,0,2,C1orf159,ripser,bottleneck,0.004390,True,Ripser on sparse kNN distance matrix,1
1,0,3,TNFRSF4,ripser,bottleneck,0.002189,True,Ripser on sparse kNN distance matrix,2
2,0,0,PLEKHN1,ripser,bottleneck,0.002183,True,Ripser on sparse kNN distance matrix,3
3,0,1,HES4,ripser,bottleneck,0.002062,True,Ripser on sparse kNN distance matrix,4
4,0,5,AJAP1,ripser,bottleneck,0.001249,True,Ripser on sparse kNN distance matrix,5
5,0,4,SMIM1,ripser,bottleneck,0.000274,True,Ripser on sparse kNN distance matrix,6


In [16]:
explainer2.grae_model_path

In [17]:
explainer.grae_model_path

'/store24/project24/ladcol_012/XplainTDA/Datasets/PBMC/CM/PBMC_GEX_GRAE_embedder.joblib'

# StratifiedXplainTDA

# Median bottleneck

In [2]:
import numpy as np
import gudhi as gd
from scipy.optimize import linear_sum_assignment

def _prep(D):
    D = np.asarray(D, float).reshape(-1, 2)
    return D[np.isfinite(D).all(1) & (D[:, 1] > D[:, 0])]

def _linf(A, B):
    return np.maximum(abs(A[:, None, 0] - B[None, :, 0]),
                      abs(A[:, None, 1] - B[None, :, 1]))

def quantile_bottleneck(D, E, q=0.5):
    D, E = _prep(D), _prep(E)
    n, m, N = len(D), len(E), len(D) + len(E)
    if N == 0:
        return 0.0

    dD, dE = (D[:, 1] - D[:, 0]) / 2, (E[:, 1] - E[:, 0]) / 2
    DE = _linf(D, E) if n and m else np.empty((n, m))
    vals = np.unique(np.r_[0, DE.ravel(), dD, dE])
    need = int(np.ceil(q * N))
    BIG = 10**9

    def good_count(t):
        valid = np.zeros((n + m, n + m), bool)
        good = np.zeros((n + m, n + m), int)

        valid[:n, :m] = True
        good[:n, :m] = 2 * (DE <= t)

        valid[np.arange(n), m + np.arange(n)] = True
        good[np.arange(n), m + np.arange(n)] = dD <= t

        valid[n + np.arange(m), np.arange(m)] = True
        good[n + np.arange(m), np.arange(m)] = dE <= t

        valid[n:, m:] = True  # diagonal-to-diagonal fillers

        r, c = linear_sum_assignment(np.where(valid, -good, BIG))
        return good[r, c].sum()

    lo, hi = 0, len(vals) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if good_count(vals[mid]) >= need:
            hi = mid
        else:
            lo = mid + 1

    return float(vals[lo])

rng = np.random.default_rng(0)

tests = [
    (
        np.array([[0.0, 1.0], [2.0, 3.0]]),
        np.array([[0.1, 1.1], [2.2, 3.1]])
    ),
    (
        np.array([[0.0, 10.0]]),
        np.array([[0.0, 1.0], [2.0, 3.0]])
    ),
    (
        np.empty((0, 2)),
        np.array([[0.0, 1.0], [2.0, 5.0]])
    ),
]

for _ in range(10):
    n, m = rng.integers(0, 8, size=2)
    D = np.c_[rng.uniform(0, 5, n), rng.uniform(5, 10, n)]
    E = np.c_[rng.uniform(0, 5, m), rng.uniform(5, 10, m)]
    tests.append((D, E))

for k, (D, E) in enumerate(tests):
    mine = quantile_bottleneck(D, E, q=1.0)
    gudhi = gd.bottleneck_distance(clean(D), clean(E), e=0)
    med = quantile_bottleneck(D, E, q=0.5)

    print(f"test {k:02d}: mine={mine:.12g}, gudhi={gudhi:.12g}, median={med:.12g}")

    assert np.isclose(mine, gudhi, atol=1e-10), (k, mine, gudhi)

print("All q=1 tests agree with GUDHI bottleneck.")

test 00: mine=0.2, gudhi=0.2, median=0.1
test 01: mine=5, gudhi=5, median=0.5
test 02: mine=1.5, gudhi=1.5, median=0.5
test 03: mine=2.6246679236, gudhi=2.6246679236, median=1.21956550464
test 04: mine=2.09008508061, gudhi=2.09008508061, median=1.24250765738
test 05: mine=2.35966195495, gudhi=2.35966195495, median=1.66525482861
test 06: mine=4.07114605062, gudhi=4.07114605062, median=1.9209556919
test 07: mine=2.613012213, gudhi=2.613012213, median=1.72556253014
test 08: mine=2.40388665404, gudhi=2.40388665404, median=1.5385106196
test 09: mine=4.75184713921, gudhi=4.75184713921, median=2.56486201675
test 10: mine=2.30514059969, gudhi=2.30514059969, median=2.02844245537
test 11: mine=3.24448076951, gudhi=3.24448076951, median=3.24448076951
test 12: mine=2.57542559326, gudhi=2.57542559326, median=1.09786309786
All q=1 tests agree with GUDHI bottleneck.
